In [0]:
%run ../../common/_utils_config

In [0]:
silver_schema = "silver_ai_test"      # the whole test runs in a throw-away schema
keep_test_schema = False              # True to inspect the test tables after the run

In [0]:
%run ../../common/_utils_reference_data

In [0]:
%run ../utils/_utils_silver_contracts

In [0]:
%run ../utils/_utils_silver_mappings

In [0]:
%run ../utils/_utils_silver_parser

In [0]:
%run ../utils/_utils_silver_quality

In [0]:
%run ../utils/_utils_silver_framework

In [0]:
import copy

failures = []


def check(description, condition):
    print(("✅ " if condition else "❌ ") + description)
    if not condition:
        failures.append(description)


assert silver_schema == "silver_ai_test", "the test must run in the test schema"
spark.sql(f"DROP SCHEMA IF EXISTS {catalog}.{silver_schema} CASCADE")

contracts = load_all_entity_contracts()
mappings = load_all_mappings()
weather_name = "open_meteo_archive__weather_observation_hourly"
weather_table = silver_table_full_name("weather_observation_hourly")


def mapping_summary(run, name):
    return next(s for s in run["summaries"] if s["mapping"] == name)

In [0]:
first = run_silver(mode="auto", only=[weather_name], file_limit=3, batch_files=1)
s1 = mapping_summary(first, weather_name)

check(f"first run ok (status {s1['status']})", s1["status"] in ("ok", "warning"))
check("3 bronze files processed", s1["files"] == 3)
check("rows in silver = valid rows", spark.table(weather_table).count() == s1["valid_rows"])
check("all valid rows inserted", s1["rows_inserted"] == s1["valid_rows"])
check("location reference loaded (52 rows)", spark.table(silver_table_full_name("location")).count() == 52)
check("data_source reference loaded", spark.table(silver_table_full_name("data_source")).count() >= 6)
check("quarantine table created and empty", spark.table(quarantine_table_name("weather_observation_hourly")).count() == 0)

first_log = spark.table(silver_run_log_table()).filter((F.col("run_id") == first["run_id"]) &
                                                       (F.col("mapping") == weather_name))
display(first_log.orderBy("batch_number"))
check("every batch logged as successful", first_log.filter(~F.col("status").isin(*success_statuses)).count() == 0)
check("quality metrics logged",
      spark.table(silver_quality_log_table()).filter(F.col("run_id") == first["run_id"]).count() > 0)

In [0]:
rows_before = spark.table(weather_table).count()
second = run_silver(mode="auto", only=[weather_name], file_limit=3, batch_files=1)
s2 = mapping_summary(second, weather_name)
check(f"second run finds no new bronze data (status {s2['status']})", s2["status"] == "no_new_data")
check("row count unchanged", spark.table(weather_table).count() == rows_before)

In [0]:
last_batch = first_log.agg(F.max("batch_number")).first()[0]
last_batch_files = first_log.filter(F.col("batch_number") == last_batch).first()["bronze_files"]
spark.sql(f"UPDATE {silver_run_log_table()} SET status = 'fail' "
          f"WHERE run_id = '{first['run_id']}' AND mapping = '{weather_name}' AND batch_number = {last_batch}")

retry = run_silver(mode="auto", only=[weather_name], file_limit=3, batch_files=1)
s3 = mapping_summary(retry, weather_name)
check(f"the failed batch is reprocessed incrementally ({s3['files']} file(s), scope {s3['scope']})",
      s3["files"] == last_batch_files and s3["scope"] == "incremental")
check("reprocessing inserts nothing (idempotent)", s3["rows_inserted"] == 0)
check("reprocessing identical data updates nothing (only real changes are written)", s3["rows_updated"] == 0)
check("row count unchanged after reprocessing", spark.table(weather_table).count() == rows_before)
check("first run was logged as full_first_load",
      {r[0] for r in first_log.select("scope").distinct().collect()} == {"full_first_load"})

In [0]:
constraints = spark.sql(f"""
    SELECT table_name, constraint_name, constraint_type
    FROM {catalog}.information_schema.table_constraints
    WHERE table_schema = '{silver_schema}'
""")
display(constraints)
rows = constraints.collect()
primary_keys = {r["table_name"] for r in rows if r["constraint_type"] == "PRIMARY KEY"}
weather_fks = [r for r in rows if r["constraint_type"] == "FOREIGN KEY" and r["table_name"] == "weather_observation_hourly"]
check("PK declared on data_source, location and weather_observation_hourly",
      {"data_source", "location", "weather_observation_hourly"} <= primary_keys)
check("weather_observation_hourly declares FKs to data_source and location", len(weather_fks) == 2)

In [0]:
broken = copy.deepcopy(mappings[weather_name])
broken["columns"]["temperature_c"]["multiply"] = 100          # every temperature out of range
rows_before = spark.table(weather_table).count()

s4 = run_mapping(broken, contracts, run_id=str(uuid.uuid4()), mode="initial", file_limit=1)
check(f"broken mapping gets status fail (got {s4['status']})", s4["status"] == "fail")
check("nothing written: row count unchanged", spark.table(weather_table).count() == rows_before)
check("nothing written: quarantine unchanged",
      spark.table(quarantine_table_name("weather_observation_hourly")).count() == 0)
fail_log = (spark.table(silver_run_log_table()).filter(F.col("status") == "fail")
            .orderBy(F.desc("finished_at")).first())
check("the failure is logged with its reason", fail_log is not None and "nothing written" in fail_log["error_message"])

In [0]:
initial = run_silver(mode="initial", only=[weather_name], file_limit=1)
s5 = mapping_summary(initial, weather_name)
check(f"initial run ok (status {s5['status']})", s5["status"] in ("ok", "warning"))
check("after initial, silver holds only the reprocessed file",
      spark.table(weather_table).count() == s5["valid_rows"])
check("rows of the other files were removed", s5["rows_deleted"] > 0)

In [0]:
# An approved change in a mapping (new version) reprocesses everything; the same version goes back to incremental
changed = copy.deepcopy(mappings[weather_name])
changed["source_record_id"] = "location_code"          # a real content change: source_record_id gets filled
rows_before = spark.table(weather_table).count()


def latest_log(name):
    return (spark.table(silver_run_log_table()).filter(F.col("mapping") == name)
            .orderBy(F.desc("finished_at")).first())


s6 = run_mapping(changed, contracts, run_id=str(uuid.uuid4()), file_limit=1)
check(f"changed mapping -> scope full_version_change (got {s6['scope']})", s6["scope"] == "full_version_change")
check("the new version is logged", latest_log(weather_name)["mapping_version"] == mapping_version(
    changed, contracts["weather_observation_hourly"]))
check("rows whose content changed are updated", s6["rows_updated"] == s6["valid_rows"] and s6["rows_inserted"] == 0)
check("source_record_id now filled", spark.table(weather_table).filter(F.col("source_record_id").isNull()).count() == 0)

s7 = run_mapping(changed, contracts, run_id=str(uuid.uuid4()), file_limit=1)
check(f"same version again -> incremental, nothing new (got {s7['scope']} / {s7['status']})",
      s7["scope"] == "incremental" and s7["status"] == "no_new_data")

s8 = run_mapping(mappings[weather_name], contracts, run_id=str(uuid.uuid4()), file_limit=1)
check("back to the approved mapping -> reprocessed again", s8["scope"] == "full_version_change"
      and s8["rows_updated"] == s8["valid_rows"])
check("row count unchanged through version changes", spark.table(weather_table).count() == rows_before)

In [0]:
check("quarantine summary view exists", spark.catalog.tableExists(quarantine_summary_view()))
display(spark.table(quarantine_summary_view()))
show_quarantine("weather_observation_hourly")

In [0]:
if not keep_test_schema:
    spark.sql(f"DROP SCHEMA IF EXISTS {catalog}.{silver_schema} CASCADE")
    print(f"🧹 dropped {catalog}.{silver_schema}")

if failures:
    raise AssertionError(f"{len(failures)} check(s) failed: {failures}")
print("✅ All silver framework checks passed")